# AlexManus 项目架构与前后端运行流程全景手册

> 更新时间：2026-03-02  
> 目标：基于当前仓库代码，讲清整个项目的设计逻辑、前后端协作方式、运行时序与排障思路。


## 这份 Notebook 的定位

这不是部署教程，也不是旧版项目介绍，而是面向当前代码的「架构说明书 + 运行流程图解」。

你可以把它当成三个层次来阅读：
1. 设计逻辑：为什么要分成 `thread / agent_run / stream` 这套结构。
2. 实际链路：一次请求从前端发出后，如何经过 FastAPI、Redis、Worker、SSE 回到 UI。
3. 运维排障：哪里最容易出问题，应该按什么顺序定位。

关键范围：
- 后端入口与路由装配：`backend/api.py`
- Agent 主链路：`backend/agent/api.py` + `backend/run_agent_background.py` + `backend/agent/run.py`
- 前端主链路：`frontend/src/components/dashboard/dashboard-content.tsx`、thread 页面与 hooks
- 认证、数据库、Redis、Sandbox 基础设施


In [ ]:
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'backend' / 'api.py').exists() and (p / 'frontend' / 'src').exists():
            return p
    raise FileNotFoundError('未找到项目根目录（需要同时存在 backend/api.py 与 frontend/src）')

ROOT = find_project_root(Path.cwd())
print('Project root:', ROOT)
print('Notebook file:', ROOT / 'AlexManus.ipynb')


## 1. 项目设计逻辑总览（先讲思想，再看代码）

当前项目的核心设计不是“页面驱动”，而是“会话运行驱动”：

1. **Thread 是会话容器**
- `thread` 代表一个持续上下文（项目下的一条对话线）。
- 线程本身保持 Agent-agnostic（线程不绑定某个固定 agent）。

2. **Agent Run 是执行实例**
- 每次真正调用模型与工具，都落成一条 `agent_run`。
- `agent_run` 有独立状态：`running/completed/stopped/failed`。

3. **流式输出与执行解耦**
- 真正执行在后台 Worker（Dramatiq）里。
- API 层负责发起、鉴权、状态查询和 SSE 转发。
- Redis 做“中间缓冲与通知总线”，把 Worker 与 API Stream 解耦。

4. **消息存储是双来源合并**
- 系统/工具/助手消息来自 `messages`。
- 用户消息在 ADK 兼容链路中来自 `events`。
- 对外统一由 `/threads/{thread_id}/messages` 合并返回。


## 2. 系统分层架构图

```text
[Browser / Next.js]
     |  REST + SSE
     v
[FastAPI api.py]
  |- /auth/*
  |- /agent/*
  |- /sandbox/*
  |- /versioning/*
  |- /triggers/*
     |
     |--- PostgreSQL (projects/threads/agent_runs/messages/events/users...)
     |
     |--- Redis
     |      |- List:   agent_run:{id}:responses
     |      |- PubSub: agent_run:{id}:new_response
     |      |- PubSub: agent_run:{id}:control
     |
     +--- Dramatiq Worker (run_agent_background)
             |
             +--- agent.run -> LLM / 工具调用 / 沙箱 / 检索服务
```


## 3. 后端启动与路由装配

`backend/api.py` 的启动逻辑（lifespan）做了这几件关键事：
1. 初始化 PostgreSQL 连接池。
2. 初始化 Redis。
3. 初始化 `agent_api`（注入 db、instance_id）。
4. 初始化 `sandbox_api`。
5. 尝试初始化 `triggers_api`。
6. 在 shutdown 时清理 agent/redis/db 资源。

该文件还统一把多个 router 挂到 `/api` 前缀下，形成单入口 API。


In [ ]:
import re

api_py = ROOT / 'backend' / 'api.py'
text = api_py.read_text(encoding='utf-8')

routers = re.findall(r"api_router\.include_router\(([^\)]+)\)", text)

print('backend/api.py 挂载的 router:')
for i, r in enumerate(routers, 1):
    print(f'{i:>2}. {r.strip()}')


### 路由模块职责（主链路视角）

| 模块 | 主要职责 |
|---|---|
| `auth_router` | 注册/登录/刷新/登出/当前用户 |
| `agent_api.router` | 会话发起、继续运行、停止、状态、流式、消息、agent 配置 |
| `sandbox_api.router` | 沙箱文件读写、沙箱生命周期、确保沙箱活跃 |
| `versioning_router` | agent 版本管理 |
| `triggers_api.router` | 触发器链路（可选业务扩展） |

说明：仓库里有更多域模块，但并非都在 `api.py` 主入口直接挂载。


## 4. 核心数据模型与状态机

| 实体 | 职责 | 关键字段（示例） | 设计要点 |
|---|---|---|---|
| `projects` | 项目容器 | `project_id`, `account_id`, `sandbox` | 承载 thread 集合与沙箱信息 |
| `threads` | 对话上下文 | `thread_id`, `project_id`, `account_id`, `metadata` | 线程与 agent 解耦 |
| `agent_runs` | 一次执行实例 | `agent_run_id`, `thread_id`, `status`, `metadata` | 运行态核心实体 |
| `messages` | 系统/助手/工具消息 | `message_id`, `thread_id`, `type`, `content` | 结构化消息源 |
| `events` | ADK 事件（含用户消息） | `session_id`, `author`, `content`, `timestamp` | 与 messages 合并输出 |
| `users` / `refresh_tokens` | 认证 | `id`, `email`, `token_hash` | JWT + refresh token 机制 |

状态流核心：
- `agent_runs.status`: `running -> completed/stopped/failed`
- 前端 `agentStatus`: `idle/connecting/running`（由 stream 回调和状态查询映射）


### 消息查询为什么要“合并两张表”

`GET /threads/{thread_id}/messages` 在后端会：
1. 分批查询 `messages`（系统、工具、助手消息）。
2. 分批查询 `events`（`author='user'` 的用户输入）。
3. 转换成统一结构后按时间排序返回。

这样做的价值：
- 兼容 ADK 事件语义。
- 对前端只暴露单一读取接口，降低 UI 复杂度。


In [ ]:
import re

agent_api_py = ROOT / 'backend' / 'agent' / 'api.py'
text = agent_api_py.read_text(encoding='utf-8')

pattern = re.compile(r'@router\.(get|post|put|delete|patch)\("([^"]+)"')
endpoints = pattern.findall(text)

focus = [
    '/agent/initiate',
    '/thread/{thread_id}/agent/start',
    '/agent-run/{agent_run_id}/stream',
    '/agent-run/{agent_run_id}/stop',
    '/thread/{thread_id}/agent-runs',
    '/threads/{thread_id}/messages',
]

print('Agent API 主链路端点扫描:')
for method, path in endpoints:
    if path in focus:
        print(f'  {method.upper():<6} {path}')


## 5. 关键时序 A：新会话发起（Dashboard）

入口在前端 `dashboard-content.tsx`：
1. 收集用户 prompt、模型参数、可选文件，封装 `FormData`。
2. 调用 `initiateAgent(formData)` -> `POST /agent/initiate`。

后端 `/agent/initiate` 关键动作：
1. 鉴权并解析模型/provider。
2. 解析 agent 配置（显式 agent 或默认 agent，必要时兜底创建默认 agent）。
3. 创建 `project` 占位记录。
4. 若有文件：创建 sandbox、上传文件、回写 `projects.sandbox`。
5. 创建 `thread`。
6. 记录首条用户消息（ADK events）。
7. 创建 `agent_run(status=running)`。
8. 注册 Redis 活跃键 `active_run:{instance}:{agent_run_id}`。
9. 投递 `run_agent_background.send(...)` 到 Dramatiq Worker。
10. 立即返回 `thread_id + agent_run_id` 给前端。


```text
Dashboard(ChatInput)
    -> POST /api/agent/initiate (FormData)
        -> create project
        -> [optional] create sandbox + upload files
        -> create thread
        -> log user event
        -> create agent_run(status=running)
        -> dramatiq: run_agent_background.send(...)
    <- {thread_id, agent_run_id}

Frontend 跳转到 thread 页面
    -> GET /api/agent-run/{id}/stream (SSE)
Worker 并行执行并持续写入 Redis
```


## 6. 关键时序 B：已有 Thread 继续对话

入口在 thread 页面（`[threadId]/page.tsx`）的 `handleSubmitMessage`：
- 同时发起两个请求（`Promise.allSettled`）：
1. `POST /threads/{thread_id}/messages`：先把用户消息入库（events 链路）。
2. `POST /thread/{thread_id}/agent/start`：为当前 thread 创建新的 `agent_run`。

`/thread/{thread_id}/agent/start` 关键细节：
1. 鉴权 + thread 访问校验。
2. 解析模型与 provider。
3. 解析/选择 agent 配置。
4. 创建 `agent_run(status=running)`。
5. 注册 Redis 活跃键。
6. 启动后台任务。

代码里有一个 `await asyncio.sleep(0.1)` 的短延迟，用于降低“用户消息写入尚未落库就启动 agent”的时序竞争。


```text
ThreadPage
  |- POST /threads/{thread_id}/messages      (user message)
  |- POST /thread/{thread_id}/agent/start    (new run)

/agent/start -> create agent_run -> send worker task
ThreadPage/useAgentStream -> streamAgent(runId)
EventSource -> /agent-run/{runId}/stream
```


## 7. 流式机制细解：Worker + Redis + SSE

核心设计：Worker 只负责“生产响应”，SSE 端点负责“把响应推给浏览器”。

Redis 关键键/频道：
- `agent_run:{id}:responses`：List，存完整响应序列。
- `agent_run:{id}:new_response`：PubSub 通知新响应到达。
- `agent_run:{id}:control`：全局控制（STOP/END_STREAM/ERROR）。
- `agent_run:{id}:control:{instance}`：实例级精准控制。

SSE 端点 `/agent-run/{id}/stream` 流程：
1. 先 `lrange` 读取 backlog，避免错过先到响应。
2. 再订阅 `new_response` 和 `control` 两个频道。
3. 收到 `new` 通知后读取增量响应并推送。
4. 收到控制信号后结束或切换错误状态。

这个设计兼顾了“实时性”和“断连可恢复”（初始回放 + 实时订阅）。


```text
Worker(run_agent_background)
  async for response in run_agent(...):
    rpush(agent_run:{id}:responses, response)
    publish(agent_run:{id}:new_response, "new")

SSE(/agent-run/{id}/stream)
  lrange(backlog) -> yield
  subscribe(new_response, control)
  on new_response -> lrange(delta) -> yield
  on control(STOP/END_STREAM/ERROR) -> close

Frontend(useAgentStream)
  EventSource.onmessage -> 解析 chunk/status/tool_call
  更新 UI 状态与消息列表
```


## 8. 停止机制（用户点击 Stop 后发生什么）

前端：`stopAgent(agentRunId)` -> `POST /agent-run/{id}/stop`

后端 `stop_agent_run(...)` 主要动作：
1. 从 Redis 拉取已积累响应（用于状态更新辅助）。
2. 更新 `agent_runs` 状态为 `stopped`（或失败场景下 `failed`）。
3. 发布 `STOP` 到全局 control channel。
4. 查找活跃实例键并向实例级 control channel 发 `STOP`。
5. 清理 Redis 响应列表。

Worker 会监听 control channel，收到 STOP 后收敛退出。


In [ ]:
import re

bg_py = ROOT / 'backend' / 'run_agent_background.py'
text = bg_py.read_text(encoding='utf-8')

patterns = sorted(set(re.findall(r"agent_run:\{agent_run_id\}:[a-z_]+", text)))
print('run_agent_background.py 中出现的 Redis key 模式:')
for p in patterns:
    print(' -', p)


## 9. 前端主链路模块职责（Thread 页面）

| 文件 | 角色 |
|---|---|
| `dashboard-content.tsx` | 新会话入口，拼装 `FormData` 调 `/agent/initiate` |
| `thread/[threadId]/page.tsx` | 会话主编排：发消息、启动 run、停止 run、拼接 UI 状态 |
| `useThreadData.ts` | 拉取 thread/project/messages/agent_runs，恢复页面上下文 |
| `useAgentStream.ts` | 管理 EventSource 生命周期，解析 chunk/tool/status，映射前端状态 |
| `useToolCalls.ts` | 从消息序列重建工具调用轨迹，驱动工具侧栏 |
| `chat-input.tsx` | 输入、模型选择、附件、停止按钮与提交行为 |
| `lib/api.ts` | 前端对后端 API 的统一封装（含 stream/start/stop/messages） |

运行时可以理解为：`page.tsx` 是 orchestration layer，hooks 分别处理“数据”“流”“工具轨迹”三类状态。


In [ ]:
important_front_files = [
    'frontend/src/components/dashboard/dashboard-content.tsx',
    'frontend/src/app/(dashboard)/projects/[projectId]/thread/[threadId]/page.tsx',
    'frontend/src/app/(dashboard)/projects/[projectId]/thread/_hooks/useThreadData.ts',
    'frontend/src/hooks/useAgentStream.ts',
    'frontend/src/app/(dashboard)/projects/[projectId]/thread/_hooks/useToolCalls.ts',
    'frontend/src/components/thread/chat-input/chat-input.tsx',
    'frontend/src/lib/api.ts',
]

from pathlib import Path

print('前端关键文件检查:')
for rel in important_front_files:
    fp = ROOT / rel
    ok = fp.exists()
    size = fp.stat().st_size if ok else 0
    print(f"[{'OK' if ok else 'MISS'}] {rel} ({size} bytes)")


## 10. 认证链路：Token 如何在前后端流动

前端认证层要点：
1. `frontend/src/lib/auth/client.ts`：自定义认证客户端，管理登录、刷新、登出、session 持久化（`localStorage`）。
2. `frontend/src/lib/supabase/client.ts`：把 auth client 与 database 兼容层组合成 supabase-like 接口。
3. `frontend/src/lib/api.ts`：调用后端前先 `getSession()`，再在请求头中带 `Authorization: Bearer <token>`。
4. SSE 场景下 EventSource 不能自定义 Header，因此把 token 放到 query 参数 `?token=...`。

后端认证层要点：
1. `get_current_user_id_from_jwt`：从 Authorization 头校验 JWT。
2. `get_user_id_from_stream_auth`：先尝试标准头，再回退 query token（专为 stream）。
3. 线程/运行访问前会走 `verify_thread_access` 或 run->thread 权限检查。


## 11. 部署拓扑与运行角色

`backend/docker-compose.yml` 中核心服务：
1. `api`：FastAPI 对外 API。
2. `worker`：Dramatiq worker，执行 `run_agent_background`。
3. `redis`：队列通信、流式通知、运行锁、短期状态。
4. `backend-postgres-new`：业务主库。

设计含义：
- API 负责同步交互（发起/查询/流式出口）。
- Worker 负责耗时推理与工具编排。
- Redis 把同步世界与异步世界连接起来。
- PostgreSQL 保存长期事实数据。


## 12. 环境变量分层（最小可运行视角）

后端必须优先配置：
1. `DATABASE_URL`
2. `JWT_SECRET_KEY`
3. `REDIS_HOST / REDIS_PORT`
4. `MODEL_TO_USE` 与对应模型 API Key（OpenAI/DeepSeek/Qwen/SiliconFlow/Ollama）

可选增强：
1. `LANGFUSE_*`（可观测性）
2. `TAVILY_API_KEY`、`FIRECRAWL_API_KEY`（联网研究能力）
3. `E2B_API_KEY` 与 `SANDBOX_TEMPLATE_*`（沙箱能力）

前端最关键：
1. `NEXT_PUBLIC_BACKEND_URL`（包含 `/api` 前缀）
2. `NEXT_PUBLIC_URL`


In [ ]:
    import re

    env_backend = (ROOT / 'backend' / '.env.example').read_text(encoding='utf-8')
    env_front = (ROOT / 'frontend' / 'env.example').read_text(encoding='utf-8')

    def parse_env_keys(text: str):
        keys = []
        for line in text.splitlines():
            s = line.strip()
            if not s or s.startswith('#') or '=' not in s:
                continue
            k = s.split('=', 1)[0].strip()
            if k:
                keys.append(k)
        return keys

    bkeys = parse_env_keys(env_backend)
    fkeys = parse_env_keys(env_front)

    print('backend/.env.example keys:', len(bkeys))
    print(', '.join(bkeys[:15]) + (' ...' if len(bkeys) > 15 else ''))
    print('
frontend/env.example keys:', len(fkeys))
    print(', '.join(fkeys))


## 13. 典型故障与排查顺序（按收益排序）

### 场景 A：前端一直转圈，没有输出
1. 先查 `agent_runs` 是否真是 `running`。
2. 再查 Redis 是否有 `agent_run:{id}:responses` 写入。
3. 查 SSE `/agent-run/{id}/stream` 是否返回并持续有 event。
4. 查 worker 是否在跑（Dramatiq 进程与日志）。

### 场景 B：Stop 点了没反应
1. 查 `/agent-run/{id}/stop` 是否返回 200。
2. 查 control channel 是否收到 `STOP`。
3. 查 worker 端 stop listener 是否在读 control channel。
4. 查 run 状态是否已更新为 `stopped/failed`。

### 场景 C：SSE 401 或断流
1. 检查 token 是否过期。
2. 检查 EventSource URL query token 是否带上。
3. 检查后端 `get_user_id_from_stream_auth` 的回退验证。

### 场景 D：文件上传/沙箱异常
1. 检查 `E2B_API_KEY` 和 template 配置。
2. 检查项目表 `sandbox` 字段是否正确回写。
3. 检查 sandbox API 的鉴权与 path 编码。


## 14. 结构化改进建议（在当前架构上演进）

1. **统一消息事实源**
- 现状是 `messages + events` 双来源合并。
- 可逐步定义单一事件模型，再由 projection 层生成前端消息视图。

2. **明确 run 状态机约束**
- 把允许转移写成显式状态机（含超时、重试、强制终止）。
- 降低“running 残留”或重复启动的边界问题。

3. **增强可观测性**
- 统一 request_id / thread_id / agent_run_id 打点。
- 把 stream 端到端延迟、首 token 时间、停止耗时做指标化。

4. **前端 stream 解析规范化**
- 把 chunk/status/tool_call 的 schema 固化并版本化，减少兼容分支。


In [ ]:
    # 一键自检：确认主链路文件和关键端点都在
    from pathlib import Path

    checks = [
        ROOT / 'backend' / 'api.py',
        ROOT / 'backend' / 'agent' / 'api.py',
        ROOT / 'backend' / 'run_agent_background.py',
        ROOT / 'frontend' / 'src' / 'lib' / 'api.ts',
        ROOT / 'frontend' / 'src' / 'hooks' / 'useAgentStream.ts',
    ]

    missing = [str(p) for p in checks if not p.exists()]
    if missing:
        raise FileNotFoundError('缺失关键文件:
' + '
'.join(missing))

    agent_api_text = (ROOT / 'backend' / 'agent' / 'api.py').read_text(encoding='utf-8')
    must_have = [
        '/agent/initiate',
        '/thread/{thread_id}/agent/start',
        '/agent-run/{agent_run_id}/stream',
        '/agent-run/{agent_run_id}/stop',
        '/threads/{thread_id}/messages',
    ]

    not_found = [ep for ep in must_have if ep not in agent_api_text]
    if not_found:
        raise AssertionError('以下关键端点未在 backend/agent/api.py 检测到:
' + '
'.join(not_found))

    print('主链路自检通过 ✅')
    print('可继续基于本手册做联调、排障和二次开发。')


## 15. 结语：建议阅读路径

如果你是新加入项目的开发者，推荐按这个顺序理解代码：
1. `backend/api.py`（入口与装配）
2. `backend/agent/api.py`（业务协议层）
3. `backend/run_agent_background.py`（异步执行与 Redis 通信）
4. `frontend/src/lib/api.ts` + `useAgentStream.ts`（前端对接与流式控制）
5. thread 页面 hooks（UI 编排）

到这一步，你就能完整解释：
- 为什么这套系统能流式实时响应；
- 为什么能支持“新建会话”和“继续会话”两条路径；
- 出问题时应从哪里下手排查。
